# Create Input Interventions

**General note:** A more detailed analysis and interpretation of the results obtained from the controlled input interventions, as well as the rationale for the chosen procedures, is provided in the final paper. This notebook focuses primarily on the implementation, validation, and documentation of the intervention procedure.

This notebook constructs controlled input variants of the held-out human-annotated test set for the shortcut analysis.

Each intervention modifies one specific source of model input while preserving the remaining information as closely as possible. Comparing model predictions on the original and modified inputs will later allow us to examine whether the models rely on:

1. the supplied target identity,
2. explicit candidate mentions in the retrieved posts, or
3. label-correlated lexical cues.

The intervention datasets are created from the held-out human-annotated test set. Whenever an intervention requires data-driven rule construction, such as identifying candidate-name variants or lexical cues, these rules are derived **exclusively from the training data** and frozen before being applied to the test set.

This separation ensures that the held-out test data do not influence the construction of the intervention rules.

A more detailed motivation and interpretation of the interventions is provided in the final paper. This notebook focuses on their implementation, validation, and reproducible construction.


## Setup and data loading

The preprocessed human-annotated subset serves as the fixed test set for all interventions.

All generated intervention datasets and audit files are stored under `data/interventions/`.

Start by loading the human annotated subset and inspecting the dataset structure:

In [1]:
import pandas as pd

# Frozen in 04_select_masking_token.ipynb
CONTEXT_MASK_TOKEN = "requ"

human_test = pd.read_parquet('../data/preprocessed/human_test.parquet')


print(f"Number of rows: {len(human_test):,}")
print(f"Columns: {human_test.columns.tolist()}")

human_test.head(3)

Number of rows: 890
Columns: ['UserId', 'TargetEntity', 'StanceLabel', 'ContextPosts']


,UserId,TargetEntity,StanceLabel,ContextPosts
0,186791,Trump,Against,"[{'Content': 'Lmao #fucktrump', 'PostTime': '2..."
1,88089,Trump,Against,"[{'Content': '""I can't get laid because of tax..."
2,254114,Trump,Against,"[{'Content': 'This has always been a problem, ..."


Inspect the target and stance-label distributions to verify the expected values in the held-out test set:

In [2]:
print("Target values:")
display(human_test["TargetEntity"].value_counts(dropna=False))

print("Stance labels:")
display(human_test["StanceLabel"].value_counts(dropna=False))

Target values:


TargetEntity
Trump     445
Harris    445
Name: count, dtype: int64

Stance labels:


StanceLabel
Against    440
Favor      235
Neither    215
Name: count, dtype: int64

Inspect one retrieved posting history to verify the nested `ContextPosts` structure used by the interventions:

In [3]:
print("First ContextPosts entry:")
print(human_test["ContextPosts"][0])

First ContextPosts entry:
[{'Content': 'Lmao #fucktrump', 'PostTime': '2024-11-25T02:39:25.551Z', 'IsAmbiguous': False}
 {'Content': 'I don’t know about anyone else but I get a kick out of people of color voting for Trump. Do you people  realize he only sees one color WHITE. He only sees one brand the RICH brand.', 'PostTime': '2024-11-24T14:56:33.890Z', 'IsAmbiguous': False}
 {'Content': 'If you are a republican and voted for Kamala I want to follow you. If you’re a republican and voted for Trump I want to say it without saying it.', 'PostTime': '2024-11-24T14:32:20.729Z', 'IsAmbiguous': False}
 {'Content': 'More people voted against Trump than voted for him.', 'PostTime': '2024-11-24T03:48:21.454Z', 'IsAmbiguous': False}
 {'Content': 'I don’t know about anyone else but I am never eating at Mc Donald’s again. They let Trump touch our fries.', 'PostTime': '2024-11-23T20:05:10.285Z', 'IsAmbiguous': False}
 {'Content': 'For as long as I live, I will never understand how anyone could have

---

## **Intervention 1: Target Masking**

The first intervention removes the identity of the supplied target while preserving the remaining model input and input structure.

For every test example:

- `TargetEntity` is replaced with an empty string (`""`).
- `ContextPosts` and all remaining example-level information remain unchanged.

Using an empty string removes the explicitly supplied target identity without introducing an additional synthetic token. The target field and surrounding input structure remain unchanged.

The intervention tests whether model predictions depend on the explicitly supplied target field. Stable predictions after masking indicate that the supplied target identity is not necessary for the prediction given the remaining retrieved context.

In [4]:
# Create the intervention from the unchanged held-out test set
target_masked_test = human_test.copy()

target_masked_test["TargetEntity"] = ""

Validate that all supplied targets were masked correctly and that no other part of the test examples was modified:

In [5]:
target_masked_test.head()

,UserId,TargetEntity,StanceLabel,ContextPosts
0,186791,,Against,"[{'Content': 'Lmao #fucktrump', 'PostTime': '2..."
1,88089,,Against,"[{'Content': '""I can't get laid because of tax..."
2,254114,,Against,"[{'Content': 'This has always been a problem, ..."
3,77504,,Against,[{'Content': 'Tomorrow morning Democracy Docke...
4,132412,,Against,"[{'Content': 'We need protection from Trump.',..."


In [6]:
# The intervention must preserve the number and ordering of examples
assert len(target_masked_test) == len(human_test)
assert target_masked_test.index.equals(human_test.index)

# Every supplied target must be removed
assert target_masked_test["TargetEntity"].eq("").all()

# All information except the supplied target must remain unchanged
unchanged_columns = [
    column for column in human_test.columns
    if column != "TargetEntity"
]

for column in unchanged_columns:
    assert target_masked_test[column].equals(human_test[column])

Finally save the target-masked test set:

In [7]:
from pathlib import Path

intervention_dir = Path("../data/interventions")
intervention_dir.mkdir(parents=True, exist_ok=True)

target_masked_test.to_parquet("../data/interventions/human_test_target_masked.parquet", index=False)

---

## **Intervention 2: Target Swapping**

The second intervention changes only the supplied target while keeping the remaining input unchanged.

For every test example:

- `Trump` is replaced with `Harris`.
- `Harris` is replaced with `Trump`.
- `ContextPosts` remain completely unchanged.

Unlike target masking, this intervention provides the model with the other candidate as the supplied target while leaving the retrieved posting history unchanged.

This intervention tests whether model predictions are sensitive to changes in the supplied target.

Important: The original StanceLabel is retained in the target-swapped dataset only to preserve the pairing with the corresponding original example. After changing the target, this label must not be interpreted as a gold label for the modified input. Target swapping is therefore evaluated through changes in model predictions and predicted class probabilities rather than classification performance against the retained label.

In [8]:
target_swapped_test = human_test.copy()

target_swap = {
    "Trump": "Harris",
    "Harris": "Trump"
}

target_swapped_test["TargetEntity"] = (target_swapped_test["TargetEntity"].map(target_swap))

Validate that all targets were swapped correctly and that no other part of the test examples was modified:

In [9]:
pd.DataFrame({
    "original_target": human_test["TargetEntity"].head(),
    "swapped_target": target_swapped_test["TargetEntity"].head(),
})

,original_target,swapped_target
0,Trump,Harris
1,Trump,Harris
2,Trump,Harris
3,Trump,Harris
4,Trump,Harris


In [10]:
# The intervention must preserve the number and ordering of examples
assert len(target_swapped_test) == len(human_test)
assert target_swapped_test.index.equals(human_test.index)

# All targets must be swapped according to the predefined mapping
expected_swapped_targets = human_test["TargetEntity"].map(target_swap)

assert expected_swapped_targets.notna().all()
assert target_swapped_test["TargetEntity"].equals(expected_swapped_targets)

# Everything except TargetEntity must remain unchanged
unchanged_columns = [
    column for column in human_test.columns
    if column != "TargetEntity"
]

for column in unchanged_columns:
    assert target_swapped_test[column].equals(human_test[column])

Finally save the target-swapped test set:

In [11]:
target_swapped_test.to_parquet("../data/interventions/human_test_target_swapped.parquet", index=False)

---

## **Intervention 3: Candidate-Mention Masking**

The third intervention removes explicit textual name-based references to Donald Trump and Kamala Harris while preserving URLs unchanged.

Candidate references identified by the frozen masking rule are replaced with the same controlled synthetic replacement token `requ`:

- `Trump` → `requ`
- `Donald Trump` → `requ`
- `Kamala` → `requ`
- `Kamala Harris` → `requ`

The same replacement token is deliberately used for both candidates. Candidate-specific replacements would preserve the identity information that the intervention is intended to remove.

References to both candidates are masked regardless of the supplied `TargetEntity`. This prevents the non-target candidate from remaining as an alternative identity cue.

When a candidate name occurs inside a larger expression, only the candidate-identifying component is replaced whenever possible. For example:

- `anti-Trump` → `anti-requ`
- `#fucktrump` → `#fuckrequ`
- `Biden-Harris` → `Biden-requ`
- `#HarrisWalz` → `#requWalz`

This preserves surrounding lexical and stance-related information while removing the explicit candidate identity.

The masking rule is constructed exclusively from the training corpus and is frozen before being applied unchanged to the held-out human-annotated test set.

The intervention therefore tests whether the models can still infer stance from indirect contextual evidence when explicit non-URL candidate-name information is removed from the retrieved posts.

### **Operational definition of a candidate mention**

For this intervention, a candidate mention is defined as an explicit name-based surface reference to Donald Trump or Kamala Harris.

The masking procedure therefore targets:

- complete candidate names,
- standalone candidate first-name and surname components,
- name-based hashtags,
- name-based handles or usernames (A handle here means a social media username/account name, typically introduced with @),
- compounds and affixed forms containing candidate-name components.

The intervention does **not** attempt to remove every expression from which candidate identity could potentially be inferred. Indirect political references such as `MAGA`, `Republican`, `Democrat`, `Vance`, `Walz`, policy terms, ideological expressions, or generic titles remain unchanged unless they explicitly contain one of the candidate-name components.

This restriction is intentional. Candidate-Mention Masking measures reliance on **explicit candidate identity**, while broader political and label-correlated expressions are examined separately in the lexical-cue intervention.

### **Training-only construction of the masking rule**

The final masking rule is derived exclusively from the training corpus rather than from the held-out test set.

The procedure is intentionally recall-oriented (find as many potentially relevant candidate mentions as possible, even if that initially includes some false positives):

1. candidate-name components are used as broad discovery anchors;
2. all training surface forms containing these anchors are collected;
3. the discovered forms are manually reviewed for systematic false positives and ambiguous uses;
4. an additional context audit identifies cases in which an otherwise valid candidate-name component belongs to another person;
5. the resulting exception and protection rules are frozen;
6. the frozen rule is subsequently applied unchanged to the held-out test set.

The review records **exceptions** to the broad matching rule rather than manually constructing a complete whitelist of candidate expressions. This allows the final matcher to generalize to previously unseen compounds or hashtags while explicitly protecting forms that were shown to be unreliable during the training-data review.

### **Load the training data**

Candidate-matching rules are derived exclusively from the preprocessed training set.

In [12]:
train = pd.read_parquet("../data/preprocessed/train.parquet")

print(f"Number of training examples: {len(train):,}")
print(f"Columns: {train.columns.tolist()}")

Number of training examples: 12,834
Columns: ['UserId', 'TargetEntity', 'StanceLabel', 'ContextPosts']


### **Extract the training post texts**

The text contents of all retrieved training posts are extracted for candidate-variant discovery.

In [13]:
# Extract the text of every retrieved training post
training_post_texts = [
    post["Content"]
    for context_posts in train["ContextPosts"]
    for post in context_posts
    if isinstance(post.get("Content"), str) and post["Content"]
]

print(f"Number of retrieved training posts: {len(training_post_texts):,}")

Number of retrieved training posts: 95,896


### **Training-only candidate-form discovery**

Four explicit candidate-name components are used as discovery anchors:

- `trump`
- `donald`
- `harris`
- `kamala`

These anchors are not themselves a final dictionary of candidate mentions. They are used to discover candidate-like surface forms occurring in the training corpus.

The search is case-insensitive and captures contiguous candidate-like surface forms containing at least one anchor. This includes ordinary name forms, hashtags, handles, compounds, affixed expressions, and alphanumeric variants.

For example:

- `Trump`
- `anti-Trump`
- `Trump2024`
- `#NeverTrump`
- `KamalaHQ`
- `Biden-Harris`

The procedure does not claim to discover arbitrary misspellings that contain none of the four anchors. It specifically discovers surface forms containing at least one exact candidate-name component.

In [14]:
search_anchors = {
    "Trump": ["trump", "donald"],
    "Harris": ["harris", "kamala"],
}

The discovery anchors are combined into a single case-insensitive regular expression.

The expression deliberately captures surrounding letters, digits, hyphens, and underscores in addition to the candidate-name anchor. This allows the review to include forms such as `anti-Trump`, `Trump2024`, or `#HarrisWalz` rather than observing only the embedded name component.

The discovery expression is intentionally broad and therefore also retrieves unrelated forms such as `McDonalds`, `Harrison`, or `trumpet`. These expected false positives are resolved during the subsequent training-data review rather than by making the initial discovery rule overly restrictive.

In [15]:
import re

# Collect all candidate-name discovery anchors
all_anchors = [anchor for anchors in search_anchors.values() for anchor in anchors]

# Escape anchors before inserting them into the regular expression
anchor_pattern = "|".join(re.escape(anchor) for anchor in all_anchors)

# Match complete surface tokens that contain at least one candidate-name anchor
# This includes ordinary tokens, hashtags, handles, compounds, and suffix variants
variant_pattern = re.compile(
    rf"(?<!\w)[#@]?[A-Za-z0-9_-]*(?:{anchor_pattern})[A-Za-z0-9_-]*(?!\w)",
    flags=re.IGNORECASE,
)

### **Collect discovered surface forms and training contexts**

Every retrieved training post is scanned with the candidate discovery pattern.

For each unique surface form, two pieces of diagnostic information are retained:

- the number of retrieved training posts in which the form occurs;
- up to three example posts containing the form.

A surface form is counted at most once per post, even if it occurs multiple times within that post.

The examples are retained solely to support the manual training-data review of false positives and ambiguous forms.

In [16]:
variant_info = {}

for text in training_post_texts:

    # Find all matching variants in the current post.
    # Using a set ensures that the same variant is counted only once per post.
    variants_in_post = {
        match.group(0).lower()
        for match in variant_pattern.finditer(text)
    }

    for variant in variants_in_post:

        # Initialize metadata for variants seen for the first time
        if variant not in variant_info:
            variant_info[variant] = {
                "post_count": 0,
                "examples": []
            }

        # Count in how many posts the variant occurs
        variant_info[variant]["post_count"] += 1

        # Store up to three example posts for later manual inspection
        if len(variant_info[variant]["examples"]) < 3:
            variant_info[variant]["examples"].append(text)


print(f"Potential surface forms discovered: {len(variant_info):,}")

Potential surface forms discovered: 782


### **Construct the candidate-form audit table**

The discovered expressions are collected in a structured audit table.

For each surface form, the table records:

- the candidate suggested by the discovery anchors;
- the observed surface form;
- the surface-form category (`variant`, `hashtag`, or `handle`);
- the number of retrieved training posts containing the form;
- up to three training examples.

If a surface form contains anchors associated with both candidates, it is labeled `BOTH`. Such forms are not considered ambiguous merely because they mention both candidates: both candidate components can be masked independently.

The audit table provides the evidence used for the subsequent manual exception review and is saved before any manual decisions are applied.

For `infer_candidate`: Each discovered surface form is assigned to the candidate whose discovery anchors occur in the expression.

If anchors associated with exactly one candidate occur, that candidate is assigned. If anchors associated with both candidates occur, the expression is labeled `BOTH`.

Because all expressions were discovered using at least one candidate-name anchor, unmatched expressions are not expected. `UNRESOLVED` is retained only as a diagnostic safeguard.

In [17]:
def infer_candidate(phrase):
    # Normalize the phrase so anchor matching is case-insensitive
    phrase = phrase.lower()

    # Identify which candidates have at least one anchor in the phrase
    matched_candidates = [
        candidate
        for candidate, anchors in search_anchors.items()
        if any(anchor in phrase for anchor in anchors)
    ]

    # Anchors associated with exactly one candidate were found
    if len(matched_candidates) == 1:
        return matched_candidates[0]

    # Anchors associated with both candidates were found
    if len(matched_candidates) > 1:
        return "BOTH"

    # This should not occur because candidate forms were discovered
    # using the same search anchors
    return "UNRESOLVED"

For: `infer_variant_type`: Each detected phrase is classified based on its prefix.  
Phrases starting with `#` are labeled as `hashtag`, phrases starting with `@` as `handle`, and all remaining phrases as `variant`.

In [18]:
def infer_variant_type(phrase):
    if phrase.startswith("#"):
        return "hashtag"

    if phrase.startswith("@"):
        return "handle"

    return "variant"

Create the table: The discovered expressions and their metadata are combined into the candidate-form audit table.

For each expression, the inferred candidate, surface-form type, frequency across retrieved training posts, and up to three example contexts are retained. The table is sorted by frequency to support the subsequent manual review.

In [19]:
# Collect detected candidate variants and their metadata
variant_rows = []

for phrase, info in variant_info.items():
    # Retrieve example posts in which the variant occurred
    examples = info["examples"]

    # Create one row per detected variant
    variant_rows.append({
        "candidate": infer_candidate(phrase),      # Trump, Harris, BOTH, or UNRESOLVED
        "phrase": phrase,                          # Detected phrase
        "type": infer_variant_type(phrase),        # hashtag, handle, or variant
        "post_count": info["post_count"],          # Number of posts containing the phrase
        "example_1": examples[0] if len(examples) > 0 else None,
        "example_2": examples[1] if len(examples) > 1 else None,
        "example_3": examples[2] if len(examples) > 2 else None,
    })

# Convert collected variants into a DataFrame,
# sort by frequency, and reset the index
candidate_variants = (
    pd.DataFrame(variant_rows)
    .sort_values(
        ["post_count", "candidate", "type", "phrase"],
        ascending=[False, True, True, True],
    )
    .reset_index(drop=True)
)

candidate_variants.head(10)

,candidate,phrase,type,post_count,example_1,example_2,example_3
0,Trump,trump,variant,35454,"I don't see how you read ""We are withholding o...","I don't see how you read ""We are withholding o...",Just in case you thought you were doing someth...
1,Harris,harris,variant,9871,"I don't see how you read ""We are withholding o...","I mean so do I, there's several people I'd wan...",All my doom and gloom aside. If Harris needs a...
2,Harris,kamala,variant,8371,"I don't see how you read ""We are withholding o...","I don't see how you read ""We are withholding o...",Kamala Harris is very good at this.
3,Trump,donald,variant,7605,"I don't see how you read ""We are withholding o...","I don't see how you read ""We are withholding o...","Not to get too political on here, but I really..."
4,Trump,#trump,hashtag,936,Trump’s Effort to Kill Off #MeToo—And the Wome...,Election 2024: Presidential results\nFormer Pr...,Dus June Kii Raat 2024 Hindi Season 2\n#Casa #...
5,Trump,trumps,variant,522,donald trumps hiring people with the one quali...,You all do realize that this point we could li...,Seems like sexual assault is a pre-requisite t...
6,Harris,#kamalaharris,hashtag,297,"“Kamala, you didn’t just run—you thundered. Yo...",I miss #KamalaHarris and #TimWalz \n\nTheir jo...,Heroine of the day: #KamalaHarris
7,Trump,mcdonald,variant,223,Trump propagandists went full North Korea over...,"If You Want McDonald to go Prison, Vote for Ha...",Ask yourself why Trump was like a dog with a b...
8,Trump,#fucktrump,hashtag,208,#FuckTrump #MAGA #MAGAts #MAGACultMorons,#FuckTrump #MAGA #MAGAts #MAGACultMorons,I only wish that tRumpublicans and the Conserv...
9,Trump,#donaldtrump,hashtag,187,Who will become the next US President? The one...,THAT'S RIGHT!!! WE WANT 'NOTHING' TO DO WITH T...,KAMALA HARRIS 2024!!!! \n🇺🇲 ❤️\n\nKamala Harri...


### **Save the candidate-form audit**

The complete set of discovered candidate-related surface forms is retained as an audit file.

The audit records the frequency and training examples underlying the matching-rule review. This makes the derivation of the final rule reproducible without hard-coding every observed candidate-related expression into the intervention itself.

In [20]:
# Save the complete training-derived candidate-form audit
candidate_variants.to_csv(
    "../data/interventions/candidate_variant_audit.csv",
    index=False,
)

### **Separate surface-form categories for review**

The discovered forms are divided into ordinary variants, hashtags, and handles because these categories exhibit different types of false positives.

Ordinary forms may contain unrelated lexical items such as `trumpet` or names such as `Harrison`. Hashtags can combine several words into a single token and therefore require separate inspection.

The same discovery rule is used for all categories; the separation is only used to make the manual audit clearer and more reproducible.

In [21]:
# Split the discovered forms into ordinary variants, hashtags, and handles
normal_variants = (candidate_variants[candidate_variants["type"] == "variant"].copy().reset_index(drop=True))

hashtags = (candidate_variants[candidate_variants["type"] == "hashtag"].copy().reset_index(drop=True))

handles = (candidate_variants[candidate_variants["type"] == "handle"].copy().reset_index(drop=True))


# Report how many discovered forms belong to each category
print(f"Normal variants: {len(normal_variants):,}")
print(f"Hashtags:        {len(hashtags):,}")
print(f"Handles:         {len(handles):,}")

Normal variants: 296
Hashtags:        486
Handles:         0


No candidate-containing handles were discovered in the retrieved training posts. Consequently, no handle-specific exception list is required for this dataset.

### **Manual review of ordinary candidate-like variants**

The ordinary candidate-like forms discovered from the training corpus are inspected together with their frequencies and example contexts.

The review records two kinds of exceptions.

**False positives** are forms for which the candidate-name anchor is part of another word, name, or place and therefore does not constitute a reference to Donald Trump or Kamala Harris. Examples include `mcdonalds`, `harrison`, and `harrisburg`.

**Ambiguous forms** are expressions that may function as candidate-related wordplay in some contexts but also have a plausible non-candidate lexical interpretation. Because the final masking rule is applied automatically and without context-specific judgement, these forms are conservatively left unchanged.

All remaining discovered forms are accepted as candidate-name references.

The decisions below were derived exclusively from the training-data normal variants audit, stored below:

In [22]:
normal_variants.to_csv(
    "../data/interventions/candidate_normal_variants_audit.csv",
    index=False,
)

In [23]:
# Expressions that contain a discovery anchor but clearly refer to another person, place, organization, or lexical item rather than Donald Trump or Kamala Harris
normal_false_positives = {
    # McDonald / McDonald's rather than Donald Trump
    "mcdonalds",
    "macdonalds",
    "mcdonalds-employee",
    "mcdonalds-",

    # Other people whose names happen to contain "donald"
    "donalds",           # Byron Donalds
    "donaldson",         # Donaldson
    "erictrump952789",   # Eric Trump rather than Donald Trump
    "maryltrump",        # Mary L. Trump rather than Donald Trump

    # Other people whose names happen to contain "harris"
    "harrison",          # Harrison Ford
    "dncjamieharrison",  # Jamie Harrison

    # Place names containing "harris"
    "harrisburg",
    "harrisonvile",
}


# Expressions that are candidate-related in at least some observed training contexts but whose surface form also has a plausible non-candidate meaning. These should not be automatically masked without additional contextual disambiguation.
normal_ambiguous = {
    
    # one observed context that may use it as candidate-directed wordplay
    "mcdonald",

    # Ordinary English verb forms that can also be used as Trump puns
    "trumped",
    "trumping",

    # Ordinary lexical items that are used as Trump-related
    # nicknames/puns in the observed political contexts
    "trumpet",
    "trumpeter",
    "trumpeteer",
    "strumpet",
    "fucktrumpet",
    "trumpery",

    # Unclear username-like construction; the surface form alone
    # does not establish a reference to Donald Trump
    "donaldsoros",
}

### **Manual review of candidate-like hashtags**

Hashtags are reviewed independently because several words can be concatenated into a single surface token.

The same decision rule is used as for ordinary variants:

- hashtags that clearly refer to something other than the two candidates are recorded as false positives;
- hashtags whose interpretation cannot be determined reliably are recorded as ambiguous;
- all remaining hashtags are accepted as candidate-name references.

Importantly, stance-bearing information surrounding the candidate name is not treated as a reason for exclusion. For example, `#fucktrump` and `#NeverTrump` remain candidate references because the candidate-identifying component can be removed while the surrounding stance information is preserved.

The decisions below were derived exclusively from the training-data hashtag audit, stored below:

In [24]:
hashtags.to_csv("../data/interventions/candidate_hashtags_audit.csv", index=False)

In [25]:
hashtag_false_positives = {
    # McDonald / McDonald's
    "#mcdonalds",
    "#mcdonald",

    # Other people
    "#byrondonalds",
    "#donaldtrumpjr",
    "#melaniatrump",
    "#harrisonford",
    "#harrison",

    # Non-candidate lexical uses
    "#trumpet",
    "#lovetrumpshate",
}

# No hashtag remained genuinely ambiguous after inspection
# of the observed training contexts.
hashtag_ambiguous = set()

### **Focused audit for non-candidate person names**

The surface-form audit alone cannot resolve every false candidate match.

For example, `Trump` is normally a valid reference to Donald Trump. However, in an expression such as `Mary Trump`, the same surname refers to another person. Similarly, the first name `Donald` may occur as part of the name of a person other than Donald Trump.

Because the final masking rule treats `Donald`, `Trump`, `Kamala`, and `Harris` as candidate-name components, a second training-only audit examines the immediate lexical context surrounding all four components.

Its purpose is specifically to identify multi-token person names in which an otherwise valid candidate-name component refers to someone other than Donald Trump or Kamala Harris.

This is distinct from the surface-form false-positive lists:

- forms such as `Harrison` or `McDonalds` can be excluded based on the discovered expression itself;
- forms such as `Mary Trump` or another person's name containing `Donald`, `Kamala`, `Trump`, or `Harris` require protection of the complete multi-token expression.

Candidate-related constructions such as `Trump administration`, `Harris campaign`, or `Trump voters` are not protected because the candidate-name component still functions as an explicit candidate-identity cue.

The following audit extracts and counts the immediate two-token contexts surrounding each of the four candidate-name components in the training corpus.

In [26]:
# Inspect immediate two-token contexts around all four candidate-name components

candidate_name_components = [
    "Donald",
    "Trump",
    "Kamala",
    "Harris",
]

component_pattern = "|".join(
    re.escape(component)
    for component in candidate_name_components
)

# Lookahead allows overlapping two-token contexts such as
# "Donald Trump" and "Trump Jr" to be collected separately.
adjacent_candidate_context_pattern = re.compile(
    rf"(?=("
    rf"\b(?:[\w.'’-]+\s+(?:{component_pattern})"
    rf"|(?:{component_pattern})\s+[\w.'’-]+)\b"
    rf"))",
    flags=re.IGNORECASE,
)

adjacent_contexts = []

for text in training_post_texts:
    for match in adjacent_candidate_context_pattern.finditer(text):
        adjacent_contexts.append(match.group(1))

adjacent_context_counts = (
    pd.Series(adjacent_contexts, dtype="string")
    .str.strip()
    .str.lower()
    .value_counts()
    .rename_axis("expression")
    .reset_index(name="count")
)

adjacent_context_counts.to_csv("../data/interventions/candidate_adjacent_context_audit.csv", index=False)


### **Protect references to other people**

Manual inspection of the focused training-context audit identified a small set of multi-token expressions in which one of the candidate-name components (`Donald`, `Trump`, `Kamala`, or `Harris`) refers to someone other than Donald Trump or Kamala Harris.

These complete expressions are protected temporarily before candidate masking and restored afterwards.

This protection is deliberately narrow. It is used only where the candidate-name component belongs to a different person or named entity. Candidate-related constructions such as `Trump administration`, `Harris campaign`, or candidate-directed name variants remain maskable.

The protection set below was derived exclusively from manual inspection of the exported training-data audit.

In [27]:
# Multi-token names containing an exact candidate-name component but referring to people other than Donald Trump or Kamala Harris.

protected_non_candidate_mentions = {
    # Other Trump family members / people named Trump
    "donald trump jr.",
    "donald trump jr",
    "trump jr.",
    "trump jr",
    "melania trump",
    "eric trump",
    "mary l. trump",
    "mary trump",
    "lara trump",
    "ivanka trump",
    "barron trump",
    "fred trump",

    # Other people named Harris found by the training audit
    "sam harris",
    "simon harris",
    "fred harris",
    "daisy harris",
    "andy harris",
    "liz harris",
    "shawn harris",
    "malcolm harris",
    "laisha harris",

    # Other non-candidate names containing Donald or Kamala
    "donald duck",
    "donald tusk",
    "donald glover",
    "donald sutherland",
    "donald rumsfeld",
    "donald sterling",
    "donald westphall",
    "kamala khan",
}

### **Compile protected non-candidate names**

The reviewed non-candidate names are converted into a single case-insensitive regular expression so that complete expressions such as `Mary Trump` or `Sam Harris` can be detected before candidate masking is applied.

The pattern is constructed from the `protected_non_candidate_mentions` set rather than written manually. Longer expressions are matched first so that more specific protected names take precedence when expressions overlap.

The boundary checks `(?<!\w)` and `(?!\w)` ensure that only complete expressions are protected rather than matching the same character sequence inside a larger word.

During masking, these expressions are temporarily replaced with internal placeholders, candidate names are masked, and the protected expressions are then restored unchanged.

In [28]:
# Compile all reviewed non-candidate names into one regex. Longer expressions are matched first to handle overlapping names.
protected_non_candidate_pattern = re.compile(
    r"(?<!\w)(?:"
    + "|".join(
        re.escape(phrase)
        for phrase in sorted(
            protected_non_candidate_mentions,
            key=len,
            reverse=True,
        )
    )
    + r")(?!\w)",
    flags=re.IGNORECASE,
)

### **Freeze candidate-masking exceptions**

The final masking rule distinguishes between two types of exceptions.

First, surface-level false positives and ambiguous forms identified during the training-data review are excluded from masking. These include lexical forms such as `mcdonalds`, `harrison`, or `trumpet`.

Second, complete names such as `Mary Trump` or `Sam Harris` are temporarily protected because the individual surname `Trump` or `Harris` would otherwise be correctly recognized as a candidate-name component.

In [29]:
# Combine confirmed false positives from normal variants and hashtags
candidate_false_positives = (normal_false_positives | hashtag_false_positives)


# Combine ambiguous expressions from normal variants and hashtags
candidate_ambiguous = (normal_ambiguous | hashtag_ambiguous)


# Neither false positives nor ambiguous forms will be masked
candidate_masking_exclusions = (candidate_false_positives | candidate_ambiguous)

print(f"False positives: {len(candidate_false_positives):,}")
print(f"Ambiguous forms: {len(candidate_ambiguous):,}")
print(f"Total excluded:  {len(candidate_masking_exclusions):,}")

False positives: 21
Ambiguous forms: 10
Total excluded:  31


### **Construct the candidate-masking function**

The frozen training-derived rules are combined into the final masking function.

The procedure first protects URLs and complete references to non-candidate people. It then masks complete candidate names and candidate-name components occurring as standalone names or within hashtags, compounds, and other candidate-related forms. Finally, the protected content is restored. URLs therefore remain unchanged by the intervention.

Complete candidate names are replaced with the frozen synthetic replacement token `requ`. For larger expressions such as hashtags or compounds, only the candidate-identifying component is replaced whenever possible so that surrounding lexical and stance-related information remains available to the model.

In [30]:
# URLs and bare domain names are preserved during masking.
url_pattern = re.compile(
    r"(?:https?://|www\.)\S+"
    r"|(?<![\w@])(?:[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?\.)+"
    r"[a-z]{2,}(?:/[^\s]*)?",
    flags=re.IGNORECASE,
)

# Match complete candidate names first.
full_candidate_name_pattern = re.compile(
    r"\b(?:"
    r"Donald(?:\s+J\.?)?\s+Trump"
    r"|"
    r"Kamala(?:\s+(?:D\.?|Devi))?\s+Harris"
    r")\b",
    flags=re.IGNORECASE,
)


# Match candidate-identifying components within larger surface forms.
candidate_component_pattern = re.compile(
    r"(?:"
    r"Donald[-_]?Trump"
    r"|Kamala[-_]?Harris"
    r"|Trump"
    r"|Donald"
    r"|Harris"
    r"|Kamala"
    r")",
    flags=re.IGNORECASE,
)


def mask_candidate_surface(match):
    """Mask candidate-name components within one surface form."""
    surface = match.group(0)

    # Reviewed false positives and ambiguous forms remain unchanged.
    if surface.lower() in candidate_masking_exclusions:
        return surface

    # Preserve surrounding information and replace only
    # candidate-identifying components.
    return candidate_component_pattern.sub(
        CONTEXT_MASK_TOKEN,
        surface,
    )


def mask_candidate_mentions(text):
    """Mask explicit non-URL candidate-name information while preserving URLs and protected non-candidate names."""

    if not isinstance(text, str):
        return text

    # 1. Temporarily protect URLs.
    protected_urls = {}

    def protect_url(match):
        placeholder = (
            f"__PROTECTED_URL_"
            f"{len(protected_urls)}__"
        )
        protected_urls[placeholder] = match.group(0)
        return placeholder

    masked_text = url_pattern.sub(
        protect_url,
        text,
    )

    # 2. Temporarily protect references to other people.
    protected_mentions = {}

    def protect_non_candidate(match):
        placeholder = (
            f"__NONCANDIDATE_PERSON_"
            f"{len(protected_mentions)}__"
        )
        protected_mentions[placeholder] = match.group(0)
        return placeholder

    masked_text = protected_non_candidate_pattern.sub(
        protect_non_candidate,
        masked_text,
    )

    # 3. Replace complete candidate names with one placeholder.
    masked_text = full_candidate_name_pattern.sub(
        CONTEXT_MASK_TOKEN,
        masked_text,
    )

    # 4. Mask remaining isolated names, hashtags,
    # compounds, and related surface forms.
    masked_text = variant_pattern.sub(
        mask_candidate_surface,
        masked_text,
    )

    # 5. Restore protected non-candidate names.
    for placeholder, original_surface in protected_mentions.items():
        masked_text = masked_text.replace(
            placeholder,
            original_surface,
        )

    # 6. Restore URLs unchanged.
    for placeholder, original_url in protected_urls.items():
        masked_text = masked_text.replace(
            placeholder,
            original_url,
        )

    return masked_text

### **Create the candidate-masked test set**

The frozen masking rule is now applied unchanged to the retrieved context posts of the human-annotated test set.

Only the `Content` field of each context post is modified. The supplied `TargetEntity`, stance label, post order, and remaining post metadata are preserved.

No masking decisions are added or revised based on the held-out test data.

In [31]:
candidate_masked_test = human_test.copy()

candidate_masked_test["ContextPosts"] = (human_test["ContextPosts"].apply(lambda context_posts: [{**post, "Content": mask_candidate_mentions(post.get("Content")),} for post in context_posts]))

Short Sanity Check to see if everything worked:

In [32]:
total_posts = 0
changed_posts = 0

for original_context, masked_context in zip(
    human_test["ContextPosts"],
    candidate_masked_test["ContextPosts"],
):
    for original_post, masked_post in zip(
        original_context,
        masked_context,
    ):
        total_posts += 1

        if (
            original_post.get("Content")
            != masked_post.get("Content")
        ):
            changed_posts += 1


print(f"Total context posts: {total_posts:,}")
print(f"Posts changed:       {changed_posts:,}")
print(
    f"Share changed:       "
    f"{changed_posts / total_posts:.2%}"
)

Total context posts: 6,829
Posts changed:       3,644
Share changed:       53.36%


In [33]:
candidate_urls = []

for context_posts in human_test["ContextPosts"]:
    for post in context_posts:
        text = post.get("Content")

        if not isinstance(text, str):
            continue

        for url_match in url_pattern.finditer(text):
            url = url_match.group(0)

            if candidate_component_pattern.search(url):
                candidate_urls.append(url)


print(
    f"URLs containing candidate-name components: "
    f"{len(candidate_urls):,}"
)

display(
    pd.Series(candidate_urls)
    .value_counts()
    .head(20)
)

URLs containing candidate-name components: 2


Trump.Time                          1
www.trumpytrout.com?mid=12243062    1
Name: count, dtype: int64

In [34]:
# Example-level information must remain unchanged
for column in human_test.columns:
    if column != "ContextPosts":
        assert candidate_masked_test[column].equals(
            human_test[column]
        )

# Context-post structure and metadata must remain unchanged
for original_context, masked_context in zip(
    human_test["ContextPosts"],
    candidate_masked_test["ContextPosts"],
):
    assert len(original_context) == len(masked_context)

    for original_post, masked_post in zip(
        original_context,
        masked_context,
    ):
        assert original_post.keys() == masked_post.keys()

        for key in original_post:
            if key != "Content":
                assert original_post[key] == masked_post[key]

def extract_urls(text):
    """Return URLs and bare domains exactly as they occur in a post."""

    if not isinstance(text, str):
        return []

    return [
        match.group(0)
        for match in url_pattern.finditer(text)
    ]


for original_context, masked_context in zip(
    human_test["ContextPosts"],
    candidate_masked_test["ContextPosts"],
):
    for original_post, masked_post in zip(
        original_context,
        masked_context,
    ):
        assert extract_urls(
            original_post.get("Content")
        ) == extract_urls(
            masked_post.get("Content")
        )

print("All URLs are preserved unchanged.")

All URLs are preserved unchanged.


In [35]:
masked_examples = []

for original_context, masked_context in zip(
    human_test["ContextPosts"],
    candidate_masked_test["ContextPosts"],
):
    for original_post, masked_post in zip(
        original_context,
        masked_context,
    ):
        original_text = original_post.get("Content")
        masked_text = masked_post.get("Content")

        if original_text != masked_text:
            masked_examples.append({
                "original": original_text,
                "masked": masked_text,
            })


masked_examples = pd.DataFrame(masked_examples)

display(
    masked_examples.sample(
        n=min(10, len(masked_examples)),
        random_state=42,
    )
)

,original,masked
415,I would rather live paycheck to paycheck than ...,I would rather live paycheck to paycheck than ...
2927,UFC just showed a Russian Op. on TV.. and the ...,UFC just showed a Russian Op. on TV.. and the ...
3194,Voters who were reluctant to back Harris becau...,Voters who were reluctant to back requ because...
298,I still say #FuckTrump and #FucktheGOP,I still say #Fuckrequ and #FucktheGOP
1874,Seen in my PA travels today. \n\nWelcome to Tr...,Seen in my PA travels today. \n\nWelcome to re...
2691,Pitch in to elect Kamala Harris and Democrats ...,Pitch in to elect requ and Democrats nationwid...
32,Trump getting his new Legion of Doom all toget...,requ getting his new Legion of Doom all together!
3313,Notice how the same folks who called Kamala Ha...,Notice how the same folks who called requ “unq...
2629,It’s time! \n\n#KamalaHarris4President \n\n💙🌊💙...,It’s time! \n\n#requ4President \n\n💙🌊💙🌊💙🌊💙🌊💙
2897,"I would like to meet a person who is like ""I w...","I would like to meet a person who is like ""I w..."


### **Document actually masked candidate surface forms**

For reproducibility, the following audit records which candidate-related surface
forms were actually modified by the frozen Candidate-Mention Masking rule in
the human-annotated test set.

This table is created only after the masking rule has been finalized. It is
therefore used for documentation rather than for developing or modifying the
masking rule.

For each modified surface form, the table records its original form, the
resulting masked form, and the number of occurrences in the test contexts.

In [36]:
masked_surface_forms = []

for context_posts in human_test["ContextPosts"]:
    for post in context_posts:
        text = post.get("Content")

        if not isinstance(text, str):
            continue

        # Temporarily protect URLs exactly as in mask_candidate_mentions().
        protected_urls = {}

        def protect_url_for_audit(match):
            placeholder = (
                f"__PROTECTED_URL_"
                f"{len(protected_urls)}__"
            )
            protected_urls[placeholder] = match.group(0)
            return placeholder

        audit_text = url_pattern.sub(
            protect_url_for_audit,
            text,
        )

        # Temporarily protect non-candidate people exactly as in mask_candidate_mentions().
        protected_mentions = {}

        def protect_non_candidate_for_audit(match):
            placeholder = (
                f"__NONCANDIDATE_PERSON_"
                f"{len(protected_mentions)}__"
            )
            protected_mentions[placeholder] = match.group(0)
            return placeholder

        audit_text = protected_non_candidate_pattern.sub(
            protect_non_candidate_for_audit,
            audit_text,
        )

        # Complete candidate names
        for match in full_candidate_name_pattern.finditer(audit_text):
            original_surface = match.group(0)

            masked_surface_forms.append({
                "original_surface": original_surface,
                "masked_surface": CONTEXT_MASK_TOKEN,
            })

        # Remove complete names temporarily so that their components are not counted a second time.
        audit_text = full_candidate_name_pattern.sub(
            CONTEXT_MASK_TOKEN,
            audit_text,
        )

        # Remaining variants / hashtags / compounds
        for match in variant_pattern.finditer(audit_text):
            original_surface = match.group(0)
            masked_surface = mask_candidate_surface(match)

            if original_surface != masked_surface:
                masked_surface_forms.append({
                    "original_surface": original_surface,
                    "masked_surface": masked_surface,
                })


candidate_masking_audit = (
    pd.DataFrame(masked_surface_forms)
    .groupby(
        ["original_surface", "masked_surface"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "count"})
    .sort_values(
        ["count", "original_surface"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

In [37]:
candidate_masking_audit.to_csv("../data/interventions/" "candidate_mention_masking_audit.csv", index=False)

The post-hoc audit documents the candidate-related forms modified by the frozen rule in the held-out test set, including hashtags and compounds. It is used only to characterize the intervention and does not inform any subsequent modification of the masking rule.

### **Save the candidate-masked test set**

The completed intervention is stored separately from the original human-annotated test set for later model evaluation.

In [38]:
candidate_masked_test.to_parquet("../data/interventions/" "human_test_candidate_mentions_masked.parquet", index=False)

---

## **Matched Random Control for Candidate-Mention Masking**

To distinguish the effect of removing explicit candidate information from the
general effect of perturbing text, Candidate-Mention Masking is accompanied by
a matched random-removal control.

For each context post, the control determines how many word-token positions are
affected by the frozen Candidate-Mention Masking rule. It then randomly selects
the same number of eligible non-candidate word-token positions from that same
post and replaces them with the same controlled synthetic replacement token requ.

Matching is performed separately within each post. This preserves which posts
in the retrieved history are perturbed and controls for the amount of removed
lexical information while maintaining post-level locality.

Explicit candidate references, reviewed protected non-candidate names, and URLs
are excluded from the pool of possible random-control positions.

For very short posts, there may be fewer eligible alternative tokens than
candidate-affected token positions. In these cases, all available eligible
tokens are removed and the remaining shortfall is recorded rather than
transferring the perturbation to another context post.

Five independently seeded control datasets are created to reduce dependence on
one particular random draw.

The control is matched on the number of affected word-token positions, not on
the exact surface transformation. Candidate-Mention Masking may replace a multi-token candidate name such as Donald Trump with one requ token, while the control independently masks the corresponding number of eligible word-token positions.

In [39]:
import numpy as np

candidate_control_placeholder = CONTEXT_MASK_TOKEN

# Word-token positions used to quantify the amount of candidate information affected by Intervention 3.
candidate_control_token_pattern = re.compile(r"(?u)\b\w+\b")

def spans_overlap(a, b):
    """Return True if two character spans overlap."""
    return a[0] < b[1] and b[0] < a[1]


def get_candidate_reference_char_spans(text):
    """
    Return character spans treated as candidate references by
    the frozen Intervention-3 masking rule.
    """

    if not isinstance(text, str):
        return []

    candidate_spans = []

    # Non-candidate names and URLs are outside the scope of
    # Candidate-Mention Masking and therefore protected.
    protected_spans = (
        [
            match.span()
            for match in protected_non_candidate_pattern.finditer(text)
        ]
        + [
            match.span()
            for match in url_pattern.finditer(text)
        ]
    )

    # Complete candidate names
    for match in full_candidate_name_pattern.finditer(text):
        span = match.span()

        if not any(
            spans_overlap(span, protected_span)
            for protected_span in protected_spans
        ):
            candidate_spans.append(span)

    # Remaining isolated names, hashtags, compounds, and variants
    for match in variant_pattern.finditer(text):
        span = match.span()

        if any(
            spans_overlap(span, protected_span)
            for protected_span in protected_spans
        ):
            continue

        surface = match.group(0)

        if mask_candidate_surface(match) == surface:
            continue

        # Record only the candidate-identifying component, not the complete surrounding surface form.
        for component_match in candidate_component_pattern.finditer(surface):
            candidate_spans.append(
                (
                    match.start() + component_match.start(),
                    match.start() + component_match.end(),
                )
            )

    return candidate_spans

The matched random control is constructed at the post level.

For each context post, the procedure first determines how many word-token positions overlap with candidate references under the frozen Candidate-Mention Masking rule. The control then randomly removes the same number of eligible non-candidate word tokens from that same post whenever possible.

Candidate references themselves, URLs, and explicitly protected non-candidate names are excluded from the pool of possible control tokens. If a post contains fewer eligible tokens than required, the resulting shortfall is retained rather than compensated for using tokens from another post.

Five control variants are generated using fixed random seeds to reduce dependence on a single random selection.

In [40]:
def get_candidate_control_info(text):
    """
    Identify candidate-affected word-token positions and eligible
    matched-control positions in one post.
    """

    if not isinstance(text, str):
        return [], set(), []

    token_matches = list(
        candidate_control_token_pattern.finditer(text)
    )

    candidate_char_spans = (
        get_candidate_reference_char_spans(text)
    )

    # Identify word tokens affected by Candidate-Mention Masking.
    candidate_indices = {
        token_idx
        for token_idx, token_match in enumerate(token_matches)
        if any(
            spans_overlap(
                token_match.span(),
                candidate_span,
            )
            for candidate_span in candidate_char_spans
        )
    }

    # Exclude information that the random control must preserve.
    protected_control_spans = (
        candidate_char_spans
        + [
            match.span()
            for match in url_pattern.finditer(text)
        ]
        + [
            match.span()
            for match in protected_non_candidate_pattern.finditer(text)
        ]
    )

    eligible_indices = []

    for token_idx, token_match in enumerate(token_matches):

        if token_idx in candidate_indices:
            continue

        if any(
            spans_overlap(
                token_match.span(),
                protected_span,
            )
            for protected_span in protected_control_spans
        ):
            continue

        eligible_indices.append(token_idx)

    return (
        token_matches,
        candidate_indices,
        eligible_indices,
    )

In [41]:
def apply_candidate_control_mask(
    text,
    token_matches,
    selected_token_indices,
):
    """
    Replace selected word-token positions with the frozen
    synthetic context-masking token.
    """

    if not isinstance(text, str):
        return text

    if not selected_token_indices:
        return text

    char_spans = [
        token_matches[int(token_idx)].span()
        for token_idx in selected_token_indices
    ]

    masked_text = text

    # Replace from right to left so that earlier character offsets remain valid.
    for start, end in sorted(
        char_spans,
        reverse=True,
    ):
        masked_text = (
            masked_text[:start]
            + candidate_control_placeholder
            + masked_text[end:]
        )

    return masked_text

In [42]:
def create_candidate_matched_random_control(df, seed):
    """
    Create one post-level matched random-control version of
    Candidate-Mention Masking.

    For each post, the control removes as many eligible
    non-candidate word-token positions as Candidate-Mention
    Masking affects in that same post.

    If a post contains too few eligible alternative tokens,
    the remaining shortfall is retained and reported rather
    than being transferred to another post.
    """

    rng = np.random.default_rng(seed)

    control_df = df.copy(deep=True)

    new_contexts = []
    audit_rows = []

    for example_idx, row in enumerate(
        df.itertuples(index=False)
    ):

        masked_context = []

        for post_idx, post in enumerate(row.ContextPosts):

            masked_post = post.copy()
            text = post.get("Content")

            (
                token_matches,
                candidate_indices,
                eligible_indices,
            ) = get_candidate_control_info(text)

            requested_tokens = len(candidate_indices)

            # Match as many candidate-affected tokens as possible within this post.
            n_remove = min(
                requested_tokens,
                len(eligible_indices),
            )

            if n_remove > 0:
                # Sample without replacement from eligible tokens in the same post.
                selected_indices = rng.choice(
                    eligible_indices,
                    size=n_remove,
                    replace=False,
                )

                selected_indices = {
                    int(token_idx)
                    for token_idx in selected_indices
                }

            else:
                selected_indices = set()

            masked_post["Content"] = (
                apply_candidate_control_mask(
                    text,
                    token_matches,
                    selected_indices,
                )
            )

            masked_context.append(masked_post)

            audit_rows.append({
                "example": example_idx,
                "post": post_idx,
                "target": row.TargetEntity,
                "requested_tokens": requested_tokens,
                "available_control_tokens": len(
                    eligible_indices
                ),
                "removed_tokens": n_remove,
                "shortfall_tokens": (
                    requested_tokens - n_remove
                ),
                "exact_match": (
                    requested_tokens == n_remove
                ),
            })

        new_contexts.append(masked_context)

    control_df["ContextPosts"] = new_contexts

    audit_df = pd.DataFrame(audit_rows)

    return control_df, audit_df

Generate five matched random-control variants using fixed seeds for reproducibility:

In [43]:
candidate_control_seeds = [1, 2, 3, 4, 5]

candidate_controls = {}
candidate_control_audits = {}

for seed in candidate_control_seeds:

    (
        candidate_controls[seed],
        candidate_control_audits[seed],
    ) = create_candidate_matched_random_control(
        human_test,
        seed=seed,
    )

### **Validate the candidate-matched random controls**

The following summary evaluates how closely each matched random-control variant reproduces the amount of information removed by Candidate-Mention Masking.

For every random seed, only posts containing at least one candidate-affected token are considered. The summary reports:

- how many posts are affected,
- how many candidate-related word-token positions should be matched,
- how many eligible control tokens were actually removed,
- how many token positions could not be matched because too few eligible alternatives were available,
- the overall percentage of requested token removals that were successfully matched, and
- the percentage of affected posts for which the requested number of removals was matched exactly.

This check is used to verify that the random controls are comparable to Candidate-Mention Masking while preserving the post-level matching constraint.

In [44]:
candidate_control_summary = []

for seed in candidate_control_seeds:

    audit = candidate_control_audits[seed]

    affected = audit[
        audit["requested_tokens"] > 0
    ]

    requested = affected[
        "requested_tokens"
    ].sum()

    removed = affected[
        "removed_tokens"
    ].sum()

    candidate_control_summary.append({
        "seed": seed,
        "affected_posts": len(affected),
        "requested_tokens": requested,
        "removed_tokens": removed,
        "shortfall_tokens": affected[
            "shortfall_tokens"
        ].sum(),
        "token_match_%": (
            removed / requested * 100
            if requested > 0
            else 100.0
        ),
        "exact_post_match_%": (
            affected["exact_match"].mean() * 100
            if len(affected) > 0
            else 100.0
        ),
    })


candidate_control_summary = pd.DataFrame(
    candidate_control_summary
)

display(candidate_control_summary)

,seed,affected_posts,requested_tokens,removed_tokens,shortfall_tokens,token_match_%,exact_post_match_%
0,1,3644,5708,5651,57,99.001402,98.682766
1,2,3644,5708,5651,57,99.001402,98.682766
2,3,3644,5708,5651,57,99.001402,98.682766
3,4,3644,5708,5651,57,99.001402,98.682766
4,5,3644,5708,5651,57,99.001402,98.682766


In [45]:
shortfall_posts = pd.concat(
    [
        audit.assign(seed=seed)
        .query("shortfall_tokens > 0")
        for seed, audit in candidate_control_audits.items()
    ],
    ignore_index=True,
)

print(
    f"Post-level shortfalls across all controls: "
    f"{len(shortfall_posts):,}"
)

Post-level shortfalls across all controls: 240


The matched controls remove eligible non-candidate word tokens from the same
context post as the corresponding candidate reference. The summary above shows
that post-level token matching is possible for nearly all affected positions.

Any remaining shortfalls occur when a post contains too few eligible alternative
tokens. These shortfalls are retained rather than transferred to other posts in
order to preserve post-level locality.

Save the five control datasets and their combined matching audit for later model evaluation.

In [46]:
candidate_control_audit_all = pd.concat(
    [
        audit.assign(seed=seed)
        for seed, audit in candidate_control_audits.items()
    ],
    ignore_index=True,
)

candidate_control_audit_all.to_csv(
    "../data/interventions/"
    "candidate_mention_control_audit.csv",
    index=False,
)

In [47]:
for seed, control_df in candidate_controls.items():

    control_df.to_parquet(
        "../data/interventions/"
        f"human_test_candidate_mentions_control_seed{seed}.parquet",
        index=False,
    )

---

## **Intervention 4: Lexical-Cue Masking**

The fourth intervention removes lexical expressions that are disproportionately
associated with particular stance labels.

Lexical cues are derived exclusively from the training set and separately for
each target and stance label:

- Trump × Favor
- Trump × Against
- Trump × Neither
- Harris × Favor
- Harris × Against
- Harris × Neither

For cue discovery, all retrieved context posts of one user-target pair are
treated as one document. Unigrams and bigrams are represented by binary document
occurrence rather than raw frequency, so repeated use of the same expression by
one user does not increase its contribution.

Candidate-name references are excluded from the resulting feature vocabulary
using the frozen Candidate-Mention Masking rule from Intervention 3.

Association strength is measured using smoothed class-vs-rest log-odds within
each target.

The following parameters define the preprocessing and smoothing settings used for the lexical comparison.

- `min_documents = 10` excludes terms that occur in fewer than 10 user-target documents for the respective target. Very rare terms are more likely to reflect individual examples or noise rather than systematic lexical differences.
- `min_class_df = 10` requires a lexical cue to occur in at least 10 user-target documents of the focal stance class before it can be selected.
- `log_odds_alpha = 0.5` adds a small pseudo-count when calculating log-odds. This prevents problems with zero counts and reduces extreme scores caused by very rare terms.
- `post_boundary_token` marks boundaries between individual posts when texts are combined. This prevents words from separate posts from being treated as if they occurred next to each other.

In [48]:
from sklearn.feature_extraction.text import CountVectorizer

# Minimum number of user-target documents in which a lexical feature must occur
min_documents = 10

# Minimum number of user-target documents within the focal stance class in which a lexical feature must occur
min_class_df = 10

# Smoothing parameter used in the log-odds calculation
log_odds_alpha = 0.5

# Special tokens used to prevent artificial lexical features across post and URL boundaries
post_boundary_token = "postboundarytoken"
url_boundary_token = "urlboundarytoken"

### **Construction of user-target context documents**

For lexical cue discovery, all retrieved context posts of each user-target pair are combined into a single document.

Only non-empty textual posts are retained. Posts are separated by a dedicated boundary token so that unigram and bigram extraction cannot create artificial bigrams across post boundaries.

URLs are replaced with a dedicated boundary token during cue discovery. This prevents technical URL components such as `https` or domain fragments from becoming lexical cues while also avoiding artificial n-grams across removed URLs. The original post texts themselves remain unchanged.

In [49]:
def build_user_target_document(context_posts):
    """Combine retrieved posts while preserving post and URL boundaries."""

    texts = []

    for post in context_posts:
        text = post.get("Content")

        if not isinstance(text, str) or not text.strip():
            continue

        text = url_pattern.sub(
            f" {url_boundary_token} ",
            text,
        )

        texts.append(text)

    return f" {post_boundary_token} ".join(texts)


# Create a training table containing the user-target pair, stance label, and corresponding lexical context document
cue_train = train[["UserId", "TargetEntity", "StanceLabel"]].copy()

# Combine all retrieved context posts for each user-target pair into one document for lexical feature extraction
cue_train["CueDocument"] = (train["ContextPosts"].apply(build_user_target_document))

cue_train.head()

,UserId,TargetEntity,StanceLabel,CueDocument
0,9,Harris,Favor,"I don't see how you read ""We are withholding o..."
1,9,Trump,Against,"I don't see how you read ""We are withholding o..."
2,10,Harris,Favor,Kamala Harris is very good at this. postbounda...
3,10,Trump,Against,AP still has Trump up postboundarytoken Not to...
4,49,Harris,Against,Let be real @anonymous\n\nJoe Scarborough is N...


### **Candidate-reference detection**

A phrase is treated as containing a candidate reference if applying the frozen masking rule changes the phrase.  
This reuses the same masking logic defined for Intervention 3 rather than introducing a separate detection rule.

In [50]:
def contains_candidate_reference(phrase):
    """Return True if the frozen Intervention-3 rule modifies the phrase."""

    return mask_candidate_mentions(phrase) != phrase

### **Construction of the lexical document-term matrix**

For each target separately, the combined context documents are transformed into a binary document-term matrix using `CountVectorizer`.

Both unigrams and bigrams are considered. A feature records whether a phrase occurs at least once in a user-target document, rather than how often it occurs within that document. This prevents repeated use of the same expression by one user from increasing its contribution.

Features occurring in fewer than `min_documents` documents for the respective target are excluded.

In [51]:
def build_target_document_matrix(target_data):

    # Initialize a vectorizer that extracts unigrams and bigrams occurring in at least the required number of documents
    vectorizer = CountVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=min_documents,
        binary=True,
    )

    # Learn the vocabulary from the cue documents and transform each document into a binary feature vector
    X = vectorizer.fit_transform(
        target_data["CueDocument"]
    )

    # Retrieve the corresponding unigram and bigram names
    phrases = vectorizer.get_feature_names_out()

    return vectorizer, X, phrases

### **Filtering lexical features**

Before lexical association scores are calculated, technical features and explicit candidate-name references are removed from the vocabulary.

Features containing the artificial post-boundary token are excluded because they arise only from the document-construction procedure. Features identified by the frozen Candidate-Mention Masking rule are also excluded because explicit candidate identity is analyzed separately in Intervention 3.

All remaining unigrams and bigrams are eligible for lexical cue scoring.

In [52]:
def lexical_feature_is_allowed(phrase):

    # Split the unigram or bigram into individual tokens
    tokens = phrase.split()

    # Exclude the artificial token used to separate individual posts
    if (
        post_boundary_token in tokens
        or url_boundary_token in tokens
    ):
        return False

    # Explicit candidate references belong to Intervention 3, not Lexical-Cue Masking.
    if contains_candidate_reference(phrase):
        return False

    # Keep all remaining lexical features
    return True

### **Stance-specific lexical cue scoring**

For each target, lexical features are evaluated separately for the `Favor`, `Against`, and `Neither` stance classes.

For a given stance, each feature is compared in a class-vs-rest setting. Because the document-term matrix is binary, `df_class` records the number of user-target documents in the focal stance class containing the phrase, while `df_other` records the corresponding number across the two remaining stance classes.

Association strength is measured using a smoothed log-odds ratio:

$$
\log \left(\frac{df_{\text{class}}+\alpha}{n_{\text{class}}-df_{\text{class}}+\alpha}\right)
-
\log \left(\frac{df_{\text{other}}+\alpha}{n_{\text{other}}-df_{\text{other}}+\alpha}\right)
$$

where $\alpha = 0.5$.

Positive values indicate that a phrase occurs proportionally more often in the focal stance class than in the remaining classes. Only positively associated phrases occurring in at least `min_class_df` focal-class documents are retained.

The remaining cues are ranked by decreasing log-odds score, with focal-class document frequency and alphabetical phrase order used as deterministic tie-breakers.

In [53]:
def score_lexical_cues_for_target(target_data):

    # All rows in target_data belong to the same target
    target = target_data["TargetEntity"].iloc[0]

    # Build the binary document-term matrix
    _, X, phrases = build_target_document_matrix(target_data)

    # Remove candidate-related and technical features.
    allowed = np.array([lexical_feature_is_allowed(phrase) for phrase in phrases])

    X = X[:, allowed]
    phrases = phrases[allowed]

    result_tables = []

    labels = ["Favor", "Against", "Neither"]

    # Compare each stance class against all remaining classes
    for label in labels:

        class_mask = (target_data["StanceLabel"].eq(label).to_numpy())

        n_class = class_mask.sum()
        n_other = (~class_mask).sum()

        # Because X is binary, summing gives document frequency.
        df_class = np.asarray(X[class_mask].sum(axis=0)).ravel()

        df_other = np.asarray(X[~class_mask].sum(axis=0)).ravel()

        log_odds = (np.log((df_class + log_odds_alpha)/ (n_class - df_class + log_odds_alpha))
            - np.log((df_other + log_odds_alpha) / (n_other - df_other + log_odds_alpha)))

        scores = pd.DataFrame({
            "target": target,
            "label": label,
            "phrase": phrases,
            "ngram": [
                len(phrase.split())
                for phrase in phrases
            ],
            "df_class": df_class,
            "df_other": df_other,
            "df_total": df_class + df_other,
            "log_odds": log_odds,
        })

        # We only want expressions positively associated
        # with this stance class.
        scores = (
            scores[
                (scores["log_odds"] > 0)
                & (scores["df_class"] >= min_class_df)
            ]
            .sort_values(
                ["log_odds", "df_class", "phrase"],
                ascending=[False, False, True],
            )
            .reset_index(drop=True)
        )

        scores["rank"] = (np.arange(len(scores)) + 1)

        result_tables.append(scores)

    return pd.concat(result_tables, ignore_index=True,)

Apply the scoring procedure independently to the training documents for each target:

In [54]:
# Calculate stance-specific lexical cue scores separately for each target
lexical_cue_scores = pd.concat(
    [score_lexical_cues_for_target(target_data.reset_index(drop=True))
        for _, target_data in cue_train.groupby("TargetEntity", sort=True)
    ], ignore_index=True)

The 25 highest-ranked cues are retained for each target–stance combination for inspection. The later intervention is constructed at two strengths, using the top 10 and top 25 cues per target–stance combination, allowing the analysis to compare a more selective with a broader lexical-cue removal.

In [55]:
# Keep only the 25 highest-ranked lexical cues for each target and stance class
top25_lexical_cues = (
    lexical_cue_scores[
        lexical_cue_scores["rank"] <= 25
    ]
    .copy()
)

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(top25_lexical_cues[["target", "label", "phrase", "ngram", "df_class", "df_other", "df_total", "log_odds", "rank",]])

,target,label,phrase,ngram,df_class,df_other,df_total,log_odds,rank
0,Harris,Favor,felons and,2,74,0,74,5.719284,1
1,Harris,Favor,way my,2,73,0,73,5.705290,2
2,Harris,Favor,against adjudicated,2,72,0,72,5.691112,3
3,Harris,Favor,and insurrection,2,72,0,72,5.691112,4
4,Harris,Favor,inciting racists,2,72,0,72,5.691112,5
5,Harris,Favor,insurrection inciting,2,72,0,72,5.691112,6
6,Harris,Favor,racists always,2,72,0,72,5.691112,7
7,Harris,Favor,rapists convicted,2,72,0,72,5.691112,8
8,Harris,Favor,vote cast,2,72,0,72,5.691112,9
9,Harris,Favor,about dem,2,37,0,37,5.015223,10


### **Manual audit of selected lexical cues**

The selected lexical cues are audited against the original training posts to examine whether highly ranked associations are driven by recurring or duplicated post texts.

Cue occurrences are identified using the same lowercase tokenization rule as `CountVectorizer`. A cue is considered present only when its tokens occur as an exact consecutive sequence within a single context post.

For each selected cue, the audit records how many user-target documents contain the cue, how many distinct matching post texts occur, and how strongly the most frequent exact post text dominates the observed occurrences.

The audit is descriptive only. It does not alter the automatically derived cue ranking or remove individual cues.

In [56]:
cue_token_pattern = re.compile(r"(?u)\b\w\w+\b")

def tokenize_for_cue_audit(text):
    """Replicate CountVectorizer's default lowercase tokenization."""
    if not isinstance(text, str):
        return []

    # Convert text to lowercase and extract all valid tokens
    return cue_token_pattern.findall(text.lower())


def audit_lexical_cue(target, label, phrase):
    """Inspect training posts containing a selected lexical cue exactly."""

    # Restrict the training data to the selected target and stance class
    subset = train[
        (train["TargetEntity"] == target)
        & (train["StanceLabel"] == label)
    ]

    # Tokenize the lexical cue using the same rule as the post text
    phrase_tokens = tokenize_for_cue_audit(phrase)
    n = len(phrase_tokens)

    matches = []

    # Inspect all context posts belonging to the selected examples
    for row in subset.itertuples():
        for post in row.ContextPosts:

            text = post.get("Content")

            if not isinstance(text, str):
                continue

            # Apply the same URL preprocessing used during cue discovery
            audit_text = url_pattern.sub(
                f" {url_boundary_token} ",
                text,
            )

            text_tokens = tokenize_for_cue_audit(audit_text)

            contains_cue = any(
                text_tokens[i:i + n] == phrase_tokens
                for i in range(len(text_tokens) - n + 1)
            )

            if contains_cue:
                matches.append({
                    "UserId": row.UserId,
                    "Content": text,
                })

    return pd.DataFrame(matches)

The audit is now applied systematically to all top-25 lexical cues.

For each cue, the resulting table compares its training-document frequency with its occurrences in the original context posts and summarizes the diversity of the matching post texts. In particular, `dominant_post_share` measures the proportion of matching posts accounted for by the single most frequent exact post text.

The complete audit is exported to `lexical_cue_audit.csv` for inspection.

In [57]:
lexical_cue_audit_rows = []

for cue in top25_lexical_cues.itertuples(index=False):

    matches = audit_lexical_cue(
        target=cue.target,
        label=cue.label,
        phrase=cue.phrase,
    )

    content_counts = matches["Content"].value_counts()

    n_users = matches["UserId"].nunique()
    n_matching_posts = len(matches)
    n_distinct_posts = matches["Content"].nunique()

    most_common_post_count = (
        int(content_counts.iloc[0])
        if len(content_counts) > 0
        else 0
    )

    lexical_cue_audit_rows.append({
        "target": cue.target,
        "label": cue.label,
        "phrase": cue.phrase,
        "rank": cue.rank,
        "df_class": cue.df_class,
        "users_with_cue": n_users,
        "matching_posts": n_matching_posts,
        "distinct_post_texts": n_distinct_posts,
        "most_common_post_count": most_common_post_count,
        "dominant_post_share": (
            most_common_post_count / n_matching_posts
            if n_matching_posts > 0
            else 0.0
        ),
    })

lexical_cue_audit = pd.DataFrame(lexical_cue_audit_rows)

lexical_cue_audit.to_csv("../data/interventions/lexical_cue_audit.csv", index=False)

In [58]:
assert (
    lexical_cue_audit["df_class"]
    == lexical_cue_audit["users_with_cue"]
).all()

print("Lexical-cue audit matches document frequencies.")

Lexical-cue audit matches document frequencies.


In [59]:
audit_for_as = audit_lexical_cue(target="Trump", label="Against", phrase="for as")

print("Users containing cue:", audit_for_as["UserId"].nunique())

print("Distinct exact post texts:", audit_for_as["Content"].nunique())

display(audit_for_as["Content"].value_counts().head(10))

Users containing cue: 215
Distinct exact post texts: 10


Content
For as long as I live, I will never understand how anyone could have chosen Trump over this.                                                                                                                                                                                              184
For as long as I live, I will never forget the cowardice of the people who let Donald Trump destroy America.                                                                                                                                                                               13
"The plain fact is that Donald Trump is not just a bad man. He is an avatar for iniquity and immorality and selfishness." We've known this for as long as Trump has been on the political scene. From the archives:                                                                         9
For as long as I live, I will never forget how the DOJ let Elon Musk give Donald Trump another presidency.                            

In [60]:
audit_felons_and = audit_lexical_cue(target="Harris", label="Favor", phrase="felons and")

print("Users containing cue:", audit_felons_and["UserId"].nunique())

print("Distinct exact post texts:", audit_felons_and["Content"].nunique())

display(audit_felons_and["Content"].value_counts().head(10))

Users containing cue: 74
Distinct exact post texts: 3


Content
I will ALWAYS be proud of the vote I cast for VP Kamala Harris for President of the United States. \n\nShe IS better than he is, in every way.\nMY vote was on the right side of history.\n\nI will always vote against adjudicated rapists, convicted felons, and insurrection inciting racists.\n\nALWAYS.    72
#Texas #HillCountry #HarrisWalz supporters regrouping for #2026 #FuckPutin #FuckTrump we will take back our country from the Oligarchs, convicted felons, and Russian agents ✊🏽                                                                                                                                  1
If we ever have an election again, I'm voting for the person who guarantees not to hand the country over to fascists, felons, and terrorists.                                                                                                                                                                    1
Name: count, dtype: int64

### **Audit of highly associated lexical cues**

The audit shows that several highly ranked lexical cues are strongly concentrated in repeated identical post texts, while others occur across a much more diverse set of posts.

The examples above illustrate this pattern for `felons and` and `for as`. These repetitions are retained because they are part of the input distribution available to the models and may themselves constitute dataset-specific lexical shortcuts.

The audit is therefore descriptive and does not alter the cue ranking or selection procedure.

The complete lexical-cue scores and the selected top-10 and top-25 cue tables are saved for reproducibility and later inspection.

In [61]:
lexical_cues_top10 = (lexical_cue_scores[lexical_cue_scores["rank"] <= 10].copy())

lexical_cues_top25 = (lexical_cue_scores[lexical_cue_scores["rank"] <= 25].copy())

In [62]:
lexical_cue_scores.to_csv("../data/interventions/lexical_cue_scores.csv", index=False)

lexical_cues_top10.to_csv("../data/interventions/lexical_cues_top10.csv",index=False)

lexical_cues_top25.to_csv("../data/interventions/lexical_cues_top25.csv", index=False)

### **Target-specific lexical cue sets**

For application to the held-out test set, the selected `Favor`, `Against`, and `Neither` cues are combined into one cue set for each target.

This is necessary because the intervention must not use the gold stance label of a test example to determine which expressions are removed. Using only the cues associated with the example's true stance would make the input modification label-dependent.

The resulting cue sets therefore depend only on the supplied target. For a Trump example, all selected Trump-associated cues are eligible for masking; for a Harris example, all selected Harris-associated cues are eligible.

Duplicate phrases occurring in more than one stance-specific ranking are included only once.

In [63]:
def build_target_cue_sets(cue_table):
    """Combine Favor, Against, and Neither cues for each target."""

     # Group the cue table by target and convert the phrases for each target into a set of unique lexical cues
    return {target: set(group["phrase"]) for target, group in cue_table.groupby("target")}


top10_cues_by_target = build_target_cue_sets(lexical_cues_top10)

top25_cues_by_target = build_target_cue_sets(lexical_cues_top25)

### **Token-consistent matching of selected lexical cues**

The selected lexical cues are converted into token sequences using the same
tokenization rule as the preceding `CountVectorizer`-based cue extraction and
manual audit.

This is necessary because `CountVectorizer` constructs n-grams from consecutive
tokens rather than from literal whitespace-separated strings. For example,
`felons and` is also extracted from `felons, and`, despite the intervening
punctuation.

The masking procedure therefore identifies cue occurrences at the token level. All word-token positions belonging to at least one selected cue are collected, and each affected token position is replaced exactly once with the frozen synthetic replacement token `requ`. This also handles overlapping lexical cues without masking the same token position multiple times.

In [64]:
lexical_placeholder = CONTEXT_MASK_TOKEN


def build_cue_sequences(cues):
    """Convert selected lexical cues into CountVectorizer-compatible token sequences."""

    sequences = {
        tuple(tokenize_for_cue_audit(cue))
        for cue in cues
    }

    # Remove possible empty token sequences
    sequences.discard(())

    # Check bigrams before unigrams
    return sorted(
        sequences,
        key=lambda sequence: (-len(sequence), sequence),
    )

In [65]:
top10_cue_sequences_by_target = {target: build_cue_sequences(cues) for target, cues in top10_cues_by_target.items()}

top25_cue_sequences_by_target = {target: build_cue_sequences(cues) for target, cues in top25_cues_by_target.items()}

### **Application of target-specific lexical-cue masking**

The previously selected lexical cues are applied to the held-out test set without modifying the original data.

For each user-target example, the masking pattern corresponding to the target is selected and applied separately to every post in `ContextPosts`. Matching lexical cues are replaced with the frozen synthetic replacement token `requ`.

Two intervention datasets are created. The first uses the union of the top-10 cues selected separately for each stance class within each target, while the second analogously uses the top-25 cues. The resulting intervention cue sets are target-specific but independent of the test example's gold stance label.

In [66]:
def get_lexical_cue_token_info(text, cue_sequences):
    """
    Return word-token matches and token positions belonging to
    selected lexical cues in one post.
    """

    if not isinstance(text, str):
        return [], set()

    token_matches = list(
        cue_token_pattern.finditer(text)
    )

    tokens = [
        match.group(0).lower()
        for match in token_matches
    ]

    # URL tokens were excluded during cue discovery and remain untouched.
    url_spans = [
        match.span()
        for match in url_pattern.finditer(text)
    ]

    url_token_indices = {
        token_idx
        for token_idx, token_match in enumerate(token_matches)
        if any(
            spans_overlap(
                token_match.span(),
                url_span,
            )
            for url_span in url_spans
        )
    }

    cue_indices = set()

    for i in range(len(tokens)):
        for cue_sequence in cue_sequences:
            n = len(cue_sequence)

            if tuple(tokens[i:i + n]) != cue_sequence:
                continue

            matched_indices = set(
                range(i, i + n)
            )

            if matched_indices & url_token_indices:
                continue

            cue_indices.update(matched_indices)

    return token_matches, cue_indices


def mask_lexical_cues_in_text(text, cue_sequences):
    """Mask every word-token position belonging to a selected lexical cue."""

    if not isinstance(text, str):
        return text

    token_matches, cue_indices = get_lexical_cue_token_info(
        text,
        cue_sequences,
    )

    if not cue_indices:
        return text

    masked_text = text

    # Replace from right to left so original character offsets remain valid.
    for token_idx in sorted(cue_indices, reverse=True):
        start, end = token_matches[token_idx].span()

        masked_text = (
            masked_text[:start]
            + lexical_placeholder
            + masked_text[end:]
        )

    return masked_text

In [67]:
def mask_lexical_cues_in_context(
    context_posts,
    target,
    cue_sequences_by_target,
):
    """Mask selected target-specific lexical cues in ContextPosts."""

    cue_sequences = cue_sequences_by_target[target]

    masked_posts = []

    for post in context_posts:

        masked_post = post.copy()

        masked_post["Content"] = mask_lexical_cues_in_text(
            post.get("Content"),
            cue_sequences,
        )

        masked_posts.append(masked_post)

    return masked_posts

In [68]:
# Create independent copies of the held-out test set
lexical_top10_masked_test = human_test.copy(deep=True)

lexical_top25_masked_test = human_test.copy(deep=True)


# Apply target-specific top-10 lexical-cue masking
lexical_top10_masked_test["ContextPosts"] = [
    mask_lexical_cues_in_context(
        context_posts=row.ContextPosts,
        target=row.TargetEntity,
        cue_sequences_by_target=top10_cue_sequences_by_target,
    )
    for row in human_test.itertuples()
]


# Apply target-specific top-25 lexical-cue masking
lexical_top25_masked_test["ContextPosts"] = [
    mask_lexical_cues_in_context(
        context_posts=row.ContextPosts,
        target=row.TargetEntity,
        cue_sequences_by_target=top25_cue_sequences_by_target,
    )
    for row in human_test.itertuples()
]

Verify that both lexical-cue interventions preserve all example-level information, context-post structure, and post metadata, changing only post content.

In [69]:
def validate_context_only_intervention(original_df, modified_df):
    """Verify that an intervention changes only post Content."""

    assert len(original_df) == len(modified_df)
    assert original_df.index.equals(modified_df.index)

    for column in original_df.columns:
        if column != "ContextPosts":
            assert modified_df[column].equals(original_df[column])

    for original_context, modified_context in zip(
        original_df["ContextPosts"],
        modified_df["ContextPosts"],
    ):
        assert len(original_context) == len(modified_context)

        for original_post, modified_post in zip(
            original_context,
            modified_context,
        ):
            assert original_post.keys() == modified_post.keys()

            for key in original_post:
                if key != "Content":
                    assert original_post[key] == modified_post[key]


validate_context_only_intervention(
    human_test,
    lexical_top10_masked_test,
)

validate_context_only_intervention(
    human_test,
    lexical_top25_masked_test,
)

print("Lexical-cue intervention structure validated.")

Lexical-cue intervention structure validated.


Summarize how strongly the two lexical-cue interventions affect the held-out test set overall and separately by target.

In [70]:
def summarize_lexical_masking(original_df, masked_df, variant):
    """Summarize how much text was affected by lexical-cue masking."""

    total_posts = 0
    masked_posts = 0
    masked_examples = 0
    total_masks = 0

    for original_row, masked_row in zip(
        original_df.itertuples(),
        masked_df.itertuples(),
    ):
        example_masks = 0

        for original_post, masked_post in zip(
            original_row.ContextPosts,
            masked_row.ContextPosts,
        ):
            total_posts += 1

            original_text = original_post.get("Content")
            masked_text = masked_post.get("Content")

            if not isinstance(original_text, str) or not isinstance(masked_text, str):
                continue

            # Count placeholders newly introduced by this intervention
            n_masks = (
                masked_text.count(lexical_placeholder)
                - original_text.count(lexical_placeholder)
            )

            if n_masks > 0:
                masked_posts += 1
                example_masks += n_masks
                total_masks += n_masks

        if example_masks > 0:
            masked_examples += 1

    return {
        "variant": variant,
        "examples": len(original_df),
        "masked_examples": masked_examples,
        "masked_examples_%": round(
            100 * masked_examples / len(original_df), 2
        ),
        "posts": total_posts,
        "masked_posts": masked_posts,
        "masked_posts_%": round(
            100 * masked_posts / total_posts, 2
        ),
        "total_masks": total_masks,
    }


masking_summary = pd.DataFrame([
    summarize_lexical_masking(
        human_test,
        lexical_top10_masked_test,
        "Top 10",
    ),
    summarize_lexical_masking(
        human_test,
        lexical_top25_masked_test,
        "Top 25",
    ),
])

display(masking_summary)

,variant,examples,masked_examples,masked_examples_%,posts,masked_posts,masked_posts_%,total_masks
0,Top 10,890,126,14.16,6829,152,2.23,705
1,Top 25,890,275,30.90,6829,374,5.48,1135


In [71]:
def summarize_masking_by_target(original_df, masked_df, variant):
    rows = []

    for target in original_df["TargetEntity"].unique():
        mask = original_df["TargetEntity"] == target

        result = summarize_lexical_masking(
            original_df.loc[mask],
            masked_df.loc[mask],
            variant,
        )

        result["target"] = target
        rows.append(result)

    return rows


masking_by_target = pd.DataFrame(
    summarize_masking_by_target(
        human_test,
        lexical_top10_masked_test,
        "Top 10",
    )
    +
    summarize_masking_by_target(
        human_test,
        lexical_top25_masked_test,
        "Top 25",
    )
)

display(masking_by_target[["variant", "target", "examples","masked_examples", "masked_examples_%", "masked_posts_%", "total_masks",]])

,variant,target,examples,masked_examples,masked_examples_%,masked_posts_%,total_masks
0,Top 10,Trump,445,78,17.53,2.91,405
1,Top 10,Harris,445,48,10.79,1.58,300
2,Top 25,Trump,445,191,42.92,7.97,708
3,Top 25,Harris,445,84,18.88,3.09,427


Save the validated top-10 and top-25 lexical-cue intervention datasets for later model inference.

In [72]:
lexical_top10_masked_test.to_parquet("../data/interventions/" "human_test_lexical_cues_top10_masked.parquet", index=False)

lexical_top25_masked_test.to_parquet("../data/interventions/" "human_test_lexical_cues_top25_masked.parquet", index=False)

---

## **Control condition for Intervention 4: Matched Random Masking**

To distinguish the specific effect of removing stance-associated lexical cues from the general effect of masking textual information, matched random control variants are created for the top-10 and top-25 lexical-cue interventions.

Random matching prioritizes the context post in which lexical-cue masking occurs. For each post, eligible non-cue word tokens from that same post are sampled first.

If a post contains too few eligible alternatives, the remaining shortfall is sampled from eligible tokens in other posts of the same user-target example. This preserves post-level locality whenever possible while keeping the overall amount of removed textual information closely matched to the intervention.

Selected lexical-cue positions, explicit candidate references, and URLs are excluded from random sampling. Tokens are never transferred across user-target examples.

Five control variants with fixed random seeds are generated for each lexical-cue condition to reduce dependence on a particular random sample.

In [73]:
def get_lexical_control_info(text, cue_sequences):
    """
    Identify lexical-cue token positions and eligible
    matched-control positions in one post.
    """

    if not isinstance(text, str):
        return [], set(), []

    # Reuse exactly the same cue detection as the lexical intervention.
    token_matches, cue_indices = get_lexical_cue_token_info(
        text,
        cue_sequences,
    )

    # Explicit candidate references and URLs cannot be random controls.
    protected_char_spans = (
        get_candidate_reference_char_spans(text)
        + [
            match.span()
            for match in url_pattern.finditer(text)
        ]
    )

    eligible_indices = []

    for token_idx, token_match in enumerate(token_matches):

        if token_idx in cue_indices:
            continue

        if any(
            spans_overlap(
                token_match.span(),
                protected_span,
            )
            for protected_span in protected_char_spans
        ):
            continue

        eligible_indices.append(token_idx)

    return (token_matches, cue_indices, eligible_indices)

In [74]:
def apply_lexical_control_mask(text, token_matches, selected_indices):
    """Replace selected control-token positions with the frozen synthetic context-masking token."""

    if not isinstance(text, str) or not selected_indices:
        return text

    masked_text = text

    # Replace from right to left so original character offsets remain valid.
    for token_idx in sorted(selected_indices, reverse=True):
        start, end = token_matches[token_idx].span()

        masked_text = (
            masked_text[:start]
            + lexical_placeholder
            + masked_text[end:]
        )

    return masked_text

In [75]:
def create_matched_random_control(df, cue_sequences_by_target, seed):
    """
    Create one hierarchically matched random-control version
    of the held-out test set.
    """

    rng = np.random.default_rng(seed)

    control_df = df.copy(deep=True)

    new_contexts = []
    audit_rows = []

    for example_idx, row in enumerate(
        df.itertuples(index=False)
    ):
        cue_sequences = cue_sequences_by_target[
            row.TargetEntity
        ]

        post_infos = []

        # First pass: match cue removals within the same post.
        for post_idx, post in enumerate(row.ContextPosts):
            text = post.get("Content")

            (
                token_matches,
                cue_indices,
                eligible_indices,
            ) = get_lexical_control_info(
                text,
                cue_sequences,
            )

            requested_tokens = len(cue_indices)

            n_local = min(
                requested_tokens,
                len(eligible_indices),
            )

            if n_local > 0:
                local_selected = {
                    int(token_idx)
                    for token_idx in rng.choice(
                        eligible_indices,
                        size=n_local,
                        replace=False,
                    )
                }
            else:
                local_selected = set()

            remaining_eligible = [
                token_idx
                for token_idx in eligible_indices
                if token_idx not in local_selected
            ]

            post_infos.append({
                "post_idx": post_idx,
                "post": post,
                "text": text,
                "token_matches": token_matches,
                "requested_tokens": requested_tokens,
                "local_selected": local_selected,
                "remaining_eligible": remaining_eligible,
            })

        requested_total = sum(
            info["requested_tokens"]
            for info in post_infos
        )

        local_removed_total = sum(
            len(info["local_selected"])
            for info in post_infos
        )

        remaining_shortfall = (
            requested_total - local_removed_total
        )

        # Second pass: fill remaining shortfalls from other posts
        # within the same user-target example.
        fallback_pool = [
            (post_idx, token_idx)
            for post_idx, info in enumerate(post_infos)
            for token_idx in info["remaining_eligible"]
        ]

        n_fallback = min(
            remaining_shortfall,
            len(fallback_pool),
        )

        if n_fallback > 0:
            fallback_positions = rng.choice(
                len(fallback_pool),
                size=n_fallback,
                replace=False,
            )

            for position in fallback_positions:
                post_idx, token_idx = fallback_pool[
                    int(position)
                ]

                post_infos[post_idx][
                    "local_selected"
                ].add(token_idx)

        masked_context = []

        for info in post_infos:
            masked_post = info["post"].copy()

            masked_post["Content"] = apply_lexical_control_mask(
                info["text"],
                info["token_matches"],
                info["local_selected"],
            )

            masked_context.append(masked_post)

        new_contexts.append(masked_context)

        removed_total = (
            local_removed_total + n_fallback
        )

        cue_affected_posts = sum(
            info["requested_tokens"] > 0
            for info in post_infos
        )

        control_modified_posts = sum(
            len(info["local_selected"]) > 0
            for info in post_infos
        )

        audit_rows.append({
            "example": example_idx,
            "target": row.TargetEntity,
            "cue_affected_posts": cue_affected_posts,
            "control_modified_posts": control_modified_posts,
            "requested_tokens": requested_total,
            "local_removed_tokens": local_removed_total,
            "fallback_removed_tokens": n_fallback,
            "removed_tokens": removed_total,
            "shortfall_tokens": (
                requested_total - removed_total
            ),
            "exact_match": (
                requested_total == removed_total
            ),
        })

    control_df["ContextPosts"] = new_contexts

    audit_df = pd.DataFrame(audit_rows)

    return control_df, audit_df

In [76]:
control_seeds = [1, 2, 3, 4, 5]

top10_controls = {}
top10_control_audits = {}

top25_controls = {}
top25_control_audits = {}

for seed in control_seeds:

    (
        top10_controls[seed],
        top10_control_audits[seed],
    ) = create_matched_random_control(
        human_test,
        top10_cue_sequences_by_target,
        seed=seed,
    )

    (
        top25_controls[seed],
        top25_control_audits[seed],
    ) = create_matched_random_control(
        human_test,
        top25_cue_sequences_by_target,
        seed=seed,
    )

The following summary verifies both the overall token-level match and how much of the random masking could be performed locally within the originally affected posts. Any remaining removals are supplied by the same-example fallback described above. 

In [77]:
def summarize_control_audits(audits, variant):
    rows = []

    for seed, audit in audits.items():
        affected = audit[
            audit["requested_tokens"] > 0
        ]

        requested = affected["requested_tokens"].sum()
        local_removed = affected["local_removed_tokens"].sum()
        fallback_removed = affected["fallback_removed_tokens"].sum()
        removed = affected["removed_tokens"].sum()

        rows.append({
            "variant": variant,
            "seed": seed,
            "affected_examples": len(affected),
            "cue_affected_posts": affected["cue_affected_posts"].sum(),
            "control_modified_posts": affected["control_modified_posts"].sum(),
            "requested_tokens": requested,
            "local_removed_tokens": local_removed,
            "fallback_removed_tokens": fallback_removed,
            "removed_tokens": removed,
            "shortfall_tokens": affected["shortfall_tokens"].sum(),
            "local_match_%": (
                local_removed / requested * 100
                if requested > 0
                else 100.0
            ),
            "token_match_%": (
                removed / requested * 100
                if requested > 0
                else 100.0
            ),
            "exact_example_match_%": (
                affected["exact_match"].mean() * 100
                if len(affected) > 0
                else 100.0
            ),
        })

    return pd.DataFrame(rows)


control_summary = pd.concat(
    [
        summarize_control_audits(
            top10_control_audits,
            "Top 10",
        ),
        summarize_control_audits(
            top25_control_audits,
            "Top 25",
        ),
    ],
    ignore_index=True,
)

display(control_summary)

,variant,seed,affected_examples,cue_affected_posts,control_modified_posts,requested_tokens,local_removed_tokens,fallback_removed_tokens,removed_tokens,shortfall_tokens,local_match_%,token_match_%,exact_example_match_%
0,Top 10,1,126,152,235,705,579,126,705,0,82.127660,100.0,100.0
1,Top 10,2,126,152,231,705,579,126,705,0,82.127660,100.0,100.0
2,Top 10,3,126,152,231,705,579,126,705,0,82.127660,100.0,100.0
3,Top 10,4,126,152,232,705,579,126,705,0,82.127660,100.0,100.0
4,Top 10,5,126,152,236,705,579,126,705,0,82.127660,100.0,100.0
5,Top 25,1,275,374,497,1135,925,210,1135,0,81.497797,100.0,100.0
6,Top 25,2,275,374,491,1135,925,210,1135,0,81.497797,100.0,100.0
7,Top 25,3,275,374,506,1135,925,210,1135,0,81.497797,100.0,100.0
8,Top 25,4,275,374,500,1135,925,210,1135,0,81.497797,100.0,100.0
9,Top 25,5,275,374,501,1135,925,210,1135,0,81.497797,100.0,100.0


The random controls match the total number of removed word-token positions exactly. Approximately 80–82% of these removals can be matched within the same post; the remaining positions are sampled from other posts of the same user-target example using the same-example fallback described above.

In [78]:
lexical_control_audit_all = pd.concat(
    [
        *[
            audit.assign(
                variant="Top 10",
                seed=seed,
            )
            for seed, audit in top10_control_audits.items()
        ],
        *[
            audit.assign(
                variant="Top 25",
                seed=seed,
            )
            for seed, audit in top25_control_audits.items()
        ],
    ],
    ignore_index=True,
)

lexical_control_audit_all.to_csv("../data/interventions/lexical_cue_control_audit.csv", index=False)

The control therefore matches the total number of removed tokens exactly while preserving post-level locality whenever sufficient eligible alternatives are available. The reported `local_match_%` quantifies how much of the control could be matched within the same post, while `fallback_removed_tokens` records the remainder drawn from other posts of the same user-target example.

In [79]:
for seed, control_df in top10_controls.items():
    control_df.to_parquet(
        "../data/interventions/"
        f"human_test_lexical_cues_top10_control_seed{seed}.parquet",
        index=False,
    )

for seed, control_df in top25_controls.items():
    control_df.to_parquet(
        "../data/interventions/"
        f"human_test_lexical_cues_top25_control_seed{seed}.parquet",
        index=False,
    )